In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline

In [16]:
df = pd.read_csv('dataset/train.csv')
df

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy
...,...,...,...,...,...,...,...,...,...,...,...
77294,77294,qp0d4n,49,2:0,0.067203,Residential,1,Not Allowed,No,11.501664,Rainy
77295,77295,qp0d4q,49,2:0,0.022859,Residential,3,Allowed,Yes,14.715254,Foggy
77296,77296,qp0d4w,49,2:0,0.141342,Residential,3,Allowed,Yes,19.678860,Sunny
77297,77297,qp0dhw,49,2:0,0.087574,Residential,1,Not Allowed,No,22.573958,Sunny


In [17]:
df.shape

(77299, 11)

In [18]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

df['hour']           = df['timestamp'].dt.hour
df['day_of_week']    = df['timestamp'].dt.dayofweek
df['month']          = df['timestamp'].dt.month
df['is_weekend']     = df['day_of_week'].isin([5, 6]).astype(int)
df['is_rush_hour']   = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df['is_night']       = df['hour'].isin(range(0, 6)).astype(int)
df['quarter']        = df['timestamp'].dt.quarter
df['week_of_year']   = df['timestamp'].dt.isocalendar().week.astype(int)

df['hour_sin']       = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']       = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']        = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']        = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin']      = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']      = np.cos(2 * np.pi * df['month'] / 12)

df['hour_x_dow']     = df['hour'] * df['day_of_week'] 
df['hour_x_weekend'] = df['hour'] * df['is_weekend']

df = df.drop(columns=['timestamp'])
df

/tmp/ipykernel_26658/4050927943.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['timestamp'] = pd.to_datetime(df['timestamp'])


,Index,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,...,quarter,week_of_year,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,hour_x_dow,hour_x_weekend
0,0,qp02z1,48,0.048804,NaN,1,Not Allowed,No,NaN,NaN,...,1,1,0.0,1.000000,0.0,1.0,0.5,0.866025,0,0
1,1,qp02zt,48,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,...,1,1,0.0,1.000000,0.0,1.0,0.5,0.866025,0,0
2,2,qp08bj,48,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,...,1,1,0.0,1.000000,0.0,1.0,0.5,0.866025,0,0
3,3,qp08gt,48,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,...,1,1,0.0,1.000000,0.0,1.0,0.5,0.866025,0,0
4,4,qp02zq,48,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,...,1,1,0.0,1.000000,0.0,1.0,0.5,0.866025,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77294,77294,qp0d4n,49,0.067203,Residential,1,Not Allowed,No,11.501664,Rainy,...,1,1,0.5,0.866025,0.0,1.0,0.5,0.866025,0,0
77295,77295,qp0d4q,49,0.022859,Residential,3,Allowed,Yes,14.715254,Foggy,...,1,1,0.5,0.866025,0.0,1.0,0.5,0.866025,0,0
77296,77296,qp0d4w,49,0.141342,Residential,3,Allowed,Yes,19.678860,Sunny,...,1,1,0.5,0.866025,0.0,1.0,0.5,0.866025,0,0
77297,77297,qp0dhw,49,0.087574,Residential,1,Not Allowed,No,22.573958,Sunny,...,1,1,0.5,0.866025,0.0,1.0,0.5,0.866025,0,0


In [19]:
df.isna().sum()

Index                0
geohash              0
day                  0
demand               0
RoadType           600
NumberofLanes        0
LargeVehicles        0
Landmarks            0
Temperature       2495
Weather            797
hour                 0
day_of_week          0
month                0
is_weekend           0
is_rush_hour         0
is_night             0
quarter              0
week_of_year         0
hour_sin             0
hour_cos             0
dow_sin              0
dow_cos              0
month_sin            0
month_cos            0
hour_x_dow           0
hour_x_weekend       0
dtype: int64

In [20]:
df = df.dropna()
df = df.drop(columns=['Index'])
print(df.isna().sum())
print(df.shape)
df

geohash           0
day               0
demand            0
RoadType          0
NumberofLanes     0
LargeVehicles     0
Landmarks         0
Temperature       0
Weather           0
hour              0
day_of_week       0
month             0
is_weekend        0
is_rush_hour      0
is_night          0
quarter           0
week_of_year      0
hour_sin          0
hour_cos          0
dow_sin           0
dow_cos           0
month_sin         0
month_cos         0
hour_x_dow        0
hour_x_weekend    0
dtype: int64
(73459, 25)


,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,...,quarter,week_of_year,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,hour_x_dow,hour_x_weekend
1,qp02zt,48,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,...,1,1,0.0,1.000000,0.0,1.0,0.5,0.866025,0,0
2,qp08bj,48,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,...,1,1,0.0,1.000000,0.0,1.0,0.5,0.866025,0,0
4,qp02zq,48,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,...,1,1,0.0,1.000000,0.0,1.0,0.5,0.866025,0,0
5,qp02zw,48,0.016262,Residential,2,Not Allowed,Yes,8.446025,Rainy,0,...,1,1,0.0,1.000000,0.0,1.0,0.5,0.866025,0,0
6,qp02zy,48,0.042247,Residential,3,Allowed,Yes,15.772408,Foggy,0,...,1,1,0.0,1.000000,0.0,1.0,0.5,0.866025,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77294,qp0d4n,49,0.067203,Residential,1,Not Allowed,No,11.501664,Rainy,2,...,1,1,0.5,0.866025,0.0,1.0,0.5,0.866025,0,0
77295,qp0d4q,49,0.022859,Residential,3,Allowed,Yes,14.715254,Foggy,2,...,1,1,0.5,0.866025,0.0,1.0,0.5,0.866025,0,0
77296,qp0d4w,49,0.141342,Residential,3,Allowed,Yes,19.678860,Sunny,2,...,1,1,0.5,0.866025,0.0,1.0,0.5,0.866025,0,0
77297,qp0dhw,49,0.087574,Residential,1,Not Allowed,No,22.573958,Sunny,2,...,1,1,0.5,0.866025,0.0,1.0,0.5,0.866025,0,0


In [21]:
df['geohash'].unique()

<StringArray>
['qp02zt', 'qp08bj', 'qp02zq', 'qp02zw', 'qp02zy', 'qp08by', 'qp08gq',
 'qp08gy', 'qp02zp', 'qp02zr',
 ...
 'qp092z', 'qp09du', 'qp097m', 'qp09y4', 'qp03zy', 'qp03yn', 'qp09vs',
 'qp098v', 'qp0d5c', 'qp0d55']
Length: 1248, dtype: str

In [22]:
df['geo_prefix'] = df['geohash'].str[:4]   

geo_stats = df.groupby('geohash')['demand'].agg(
    geo_mean_demand='mean',
    geo_median_demand='median',
    geo_std_demand='std'
).reset_index()

df = df.merge(geo_stats, on='geohash', how='left')

df['geo_mean_demand']   = df['geo_mean_demand'].fillna(df['demand'].mean())
df['geo_median_demand'] = df['geo_median_demand'].fillna(df['demand'].median())
df['geo_std_demand']    = df['geo_std_demand'].fillna(df['demand'].std())

In [23]:
df['temp_bin'] = pd.cut(df['Temperature'],
                         bins=[-np.inf, 0, 10, 20, 30, np.inf],
                         labels=['freezing', 'cold', 'mild', 'warm', 'hot'])

In [24]:
X = df.drop(columns='demand')
y = df['demand']

df

,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,...,dow_cos,month_sin,month_cos,hour_x_dow,hour_x_weekend,geo_prefix,geo_mean_demand,geo_median_demand,geo_std_demand,temp_bin
0,qp02zt,48,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,...,1.0,0.5,0.866025,0,0,qp02,0.209034,0.215480,0.102974,hot
1,qp08bj,48,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,...,1.0,0.5,0.866025,0,0,qp08,0.131613,0.123451,0.074360,warm
2,qp02zq,48,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,...,1.0,0.5,0.866025,0,0,qp02,0.030793,0.026722,0.026321,mild
3,qp02zw,48,0.016262,Residential,2,Not Allowed,Yes,8.446025,Rainy,0,...,1.0,0.5,0.866025,0,0,qp02,0.492461,0.570624,0.299060,cold
4,qp02zy,48,0.042247,Residential,3,Allowed,Yes,15.772408,Foggy,0,...,1.0,0.5,0.866025,0,0,qp02,0.136552,0.123432,0.072716,mild
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73454,qp0d4n,49,0.067203,Residential,1,Not Allowed,No,11.501664,Rainy,2,...,1.0,0.5,0.866025,0,0,qp0d,0.026405,0.029218,0.015699,mild
73455,qp0d4q,49,0.022859,Residential,3,Allowed,Yes,14.715254,Foggy,2,...,1.0,0.5,0.866025,0,0,qp0d,0.138692,0.103773,0.116414,mild
73456,qp0d4w,49,0.141342,Residential,3,Allowed,Yes,19.678860,Sunny,2,...,1.0,0.5,0.866025,0,0,qp0d,0.113167,0.092020,0.084941,mild
73457,qp0dhw,49,0.087574,Residential,1,Not Allowed,No,22.573958,Sunny,2,...,1.0,0.5,0.866025,0,0,qp0d,0.074273,0.071481,0.039762,warm


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [26]:
categorical_cols = [
    'geohash',
    'geo_prefix',       
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'temp_bin',         
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', TargetEncoder(), categorical_cols)
    ],
    remainder='passthrough'
)

In [27]:
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    tree_method='hist',
    device='cuda', 
    eval_metric='rmse',
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb)
])

In [28]:
param_grid = {
    'model__n_estimators':        [400, 800, 1200],
    'model__max_depth':           [4, 6, 8, 10],
    'model__min_child_weight':    [1, 3, 5, 7],
    'model__max_leaves':          [0, 31, 63], 

    'model__subsample':           [0.7, 0.8, 0.9],
    'model__colsample_bytree':    [0.7, 0.8, 1.0],
    'model__colsample_bylevel':   [0.7, 1.0],       
    'model__colsample_bynode':    [0.7, 1.0],       

    'model__learning_rate':       [0.005, 0.01, 0.05],

    'model__reg_alpha':           [0, 0.01, 0.1, 0.5],   
    'model__reg_lambda':          [0.5, 1, 2, 5],         
    'model__gamma':               [0, 0.1, 0.5, 1],       

    'model__booster':           ['gbtree', 'dart'],
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,
    n_iter=100,         
    scoring='r2',
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

X_train_preprocessed = preprocessor.fit_transform(X_train, y_train)
X_test_preprocessed  = preprocessor.transform(X_test)

random_search.fit(
    X_train, y_train,
    model__eval_set=[(X_test_preprocessed, y_test)], 
    model__verbose=False
)

print("Best Params:", random_search.best_params_)
print("Best CV R²:", random_search.best_score_)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


/home/akshat/gridlock/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [16:52:33] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/home/akshat/gridlock/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [16:52:33] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordin

Best Params: {'model__subsample': 0.9, 'model__reg_lambda': 1, 'model__reg_alpha': 0.01, 'model__n_estimators': 1200, 'model__min_child_weight': 5, 'model__max_leaves': 0, 'model__max_depth': 8, 'model__learning_rate': 0.05, 'model__gamma': 0, 'model__colsample_bytree': 0.8, 'model__colsample_bynode': 1.0, 'model__colsample_bylevel': 0.7, 'model__booster': 'gbtree'}
Best CV R²: 0.9487973933321998


In [29]:
y_pred = random_search.best_estimator_.predict(X_test)
print("Test R²:", r2_score(y_test, y_pred))

Test R²: 0.9545074567642381


/home/akshat/gridlock/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [19:59:24] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [30]:
df_test    = pd.read_csv('dataset/test.csv')
test_index = df_test['Index']
df_test    = df_test.drop(columns=['Index'])
df_test

,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN
1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy
2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny
3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy
4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy
...,...,...,...,...,...,...,...,...,...
41773,qp0d4q,49,13:45,Street,1,Not Allowed,Yes,19.588991,Sunny
41774,qp0d4w,49,13:45,Residential,2,Not Allowed,Yes,10.735538,Rainy
41775,qp0dhq,49,13:45,Residential,2,Not Allowed,Yes,13.223750,Rainy
41776,qp0dhw,49,13:45,Residential,2,Not Allowed,Yes,12.510917,Rainy


In [31]:
df_test.isna().sum()

geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      1349
Weather           431
dtype: int64

In [32]:
for col in ['RoadType', 'Weather']:
    df_test[col] = df_test[col].fillna(X_train[col].mode()[0])
df_test['Temperature'] = df_test['Temperature'].fillna(X_train['Temperature'].median())

df_test.isna().sum()

geohash          0
day              0
timestamp        0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
dtype: int64

In [34]:
df_test['timestamp'] = pd.to_datetime(df_test['timestamp'])

df_test['hour']           = df_test['timestamp'].dt.hour
df_test['day_of_week']    = df_test['timestamp'].dt.dayofweek
df_test['month']          = df_test['timestamp'].dt.month
df_test['is_weekend']     = df_test['day_of_week'].isin([5, 6]).astype(int)
df_test['is_rush_hour']   = df_test['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df_test['is_night']       = df_test['hour'].isin(range(0, 6)).astype(int)
df_test['quarter']        = df_test['timestamp'].dt.quarter
df_test['week_of_year']   = df_test['timestamp'].dt.isocalendar().week.astype(int)

df_test['hour_sin']       = np.sin(2 * np.pi * df_test['hour'] / 24)
df_test['hour_cos']       = np.cos(2 * np.pi * df_test['hour'] / 24)
df_test['dow_sin']        = np.sin(2 * np.pi * df_test['day_of_week'] / 7)
df_test['dow_cos']        = np.cos(2 * np.pi * df_test['day_of_week'] / 7)
df_test['month_sin']      = np.sin(2 * np.pi * df_test['month'] / 12)
df_test['month_cos']      = np.cos(2 * np.pi * df_test['month'] / 12)

df_test['hour_x_dow']     = df_test['hour'] * df_test['day_of_week'] 
df_test['hour_x_weekend'] = df_test['hour'] * df_test['is_weekend']

df_test = df_test.drop(columns=['timestamp'])
df_test

,geohash,day,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,day_of_week,...,quarter,week_of_year,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,hour_x_dow,hour_x_weekend
0,qp02z1,49,Residential,1,Not Allowed,No,16.380041,Sunny,2,3,...,2,22,0.500000,0.866025,0.433884,-0.900969,0.5,-0.866025,6,0
1,qp02z9,49,Residential,1,Not Allowed,No,6.476213,Snowy,2,3,...,2,22,0.500000,0.866025,0.433884,-0.900969,0.5,-0.866025,6,0
2,qp02yf,49,Residential,3,Allowed,Yes,22.318203,Sunny,2,3,...,2,22,0.500000,0.866025,0.433884,-0.900969,0.5,-0.866025,6,0
3,qp02z6,49,Residential,2,Not Allowed,Yes,16.380041,Rainy,2,3,...,2,22,0.500000,0.866025,0.433884,-0.900969,0.5,-0.866025,6,0
4,qp02zd,49,Residential,1,Not Allowed,No,18.266162,Foggy,2,3,...,2,22,0.500000,0.866025,0.433884,-0.900969,0.5,-0.866025,6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41773,qp0d4q,49,Street,1,Not Allowed,Yes,19.588991,Sunny,13,3,...,2,22,-0.258819,-0.965926,0.433884,-0.900969,0.5,-0.866025,39,0
41774,qp0d4w,49,Residential,2,Not Allowed,Yes,10.735538,Rainy,13,3,...,2,22,-0.258819,-0.965926,0.433884,-0.900969,0.5,-0.866025,39,0
41775,qp0dhq,49,Residential,2,Not Allowed,Yes,13.223750,Rainy,13,3,...,2,22,-0.258819,-0.965926,0.433884,-0.900969,0.5,-0.866025,39,0
41776,qp0dhw,49,Residential,2,Not Allowed,Yes,12.510917,Rainy,13,3,...,2,22,-0.258819,-0.965926,0.433884,-0.900969,0.5,-0.866025,39,0


In [36]:
df_test['geo_prefix'] = df_test['geohash'].str[:4]   

geo_stats = df.groupby('geohash')['demand'].agg(
    geo_mean_demand='mean',
    geo_median_demand='median',
    geo_std_demand='std'
).reset_index()

df_test = df_test.merge(geo_stats, on='geohash', how='left')

global_mean   = df['demand'].mean()
global_median = df['demand'].median()
global_std    = df['demand'].std()

for df in [df, df_test]:
    df['geo_mean_demand']   = df['geo_mean_demand'].fillna(global_mean)
    df['geo_median_demand'] = df['geo_median_demand'].fillna(global_median)
    df['geo_std_demand']    = df['geo_std_demand'].fillna(global_std)

In [37]:
df_test['temp_bin'] = pd.cut(df_test['Temperature'],
                         bins=[-np.inf, 0, 10, 20, 30, np.inf],
                         labels=['freezing', 'cold', 'mild', 'warm', 'hot'])

df

,geohash,day,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,day_of_week,...,dow_cos,month_sin,month_cos,hour_x_dow,hour_x_weekend,geo_prefix,geo_mean_demand,geo_median_demand,geo_std_demand,temp_bin
0,qp02z1,49,Residential,1,Not Allowed,No,16.380041,Sunny,2,3,...,-0.900969,0.5,-0.866025,6,0,qp02,0.039774,0.034761,0.030564,mild
1,qp02z9,49,Residential,1,Not Allowed,No,6.476213,Snowy,2,3,...,-0.900969,0.5,-0.866025,6,0,qp02,0.031442,0.027846,0.023999,cold
2,qp02yf,49,Residential,3,Allowed,Yes,22.318203,Sunny,2,3,...,-0.900969,0.5,-0.866025,6,0,qp02,0.029433,0.029433,0.142083,warm
3,qp02z6,49,Residential,2,Not Allowed,Yes,16.380041,Rainy,2,3,...,-0.900969,0.5,-0.866025,6,0,qp02,0.039944,0.027554,0.035554,mild
4,qp02zd,49,Residential,1,Not Allowed,No,18.266162,Foggy,2,3,...,-0.900969,0.5,-0.866025,6,0,qp02,0.054595,0.051226,0.034540,mild
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41773,qp0d4q,49,Street,1,Not Allowed,Yes,19.588991,Sunny,13,3,...,-0.900969,0.5,-0.866025,39,0,qp0d,0.138692,0.103773,0.116414,mild
41774,qp0d4w,49,Residential,2,Not Allowed,Yes,10.735538,Rainy,13,3,...,-0.900969,0.5,-0.866025,39,0,qp0d,0.113167,0.092020,0.084941,mild
41775,qp0dhq,49,Residential,2,Not Allowed,Yes,13.223750,Rainy,13,3,...,-0.900969,0.5,-0.866025,39,0,qp0d,0.018260,0.014739,0.013804,mild
41776,qp0dhw,49,Residential,2,Not Allowed,Yes,12.510917,Rainy,13,3,...,-0.900969,0.5,-0.866025,39,0,qp0d,0.074273,0.071481,0.039762,mild


In [38]:
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,

    # GPU
    tree_method='hist',
    device='cuda',

    # Best params from GridSearchCV
    booster='gbtree',
    n_estimators=1200,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,

    colsample_bytree=0.8,
    colsample_bylevel=0.7,
    colsample_bynode=1.0,

    min_child_weight=5,
    gamma=0,

    reg_alpha=0.01,
    reg_lambda=1,

    max_leaves=0
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb)
])

In [39]:
pipeline.fit(X_train, y_train)
print("Train R²:", r2_score(y_test, pipeline.predict(X_test)))

Train R²: 0.9546638396078033


In [40]:
y_pred = pipeline.predict(df_test)

submission = pd.DataFrame({
    'Index':  test_index,
    'demand': y_pred
})

submission

,Index,demand
0,0,0.046556
1,1,0.037800
2,2,-0.006264
3,3,0.020797
4,4,0.051931
...,...,...
41773,41773,0.300412
41774,41774,0.154686
41775,41775,0.014127
41776,41776,0.113789


In [41]:
submission.to_csv('submission.csv', index=False)